# Crys-JEPA + DFT Superconductivity Training on Colab

Notebook pour entraîner les variantes **DFT seul** et **DFT-JEPA** sur GPU Colab.

Ce notebook suppose que tu as soit :

1. un dossier Drive `/content/drive/MyDrive/Crys_JEPA` contenant le projet et les données ;
2. ou une archive `/content/drive/MyDrive/Crys_JEPA.zip` contenant le projet ;
3. optionnellement une archive `/content/drive/MyDrive/3DSC_data.zip` contenant `data/raw/3DSC_MP.csv` et `data/raw/cifs/`.

Les entraînements utilisent :

- `configs/ablation/dft.yaml`
- `configs/ablation/crys_jepa_dft.yaml`

Les checkpoints seront sauvegardés dans Drive à la fin.

In [ ]:
# Vérifier le GPU Colab
!nvidia-smi

Sun Jun 28 15:07:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Monter Google Drive
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Configuration utilisateur
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_NAME = 'Crys_JEPA'
PROJECT_DIR = Path('/content') / PROJECT_NAME

# Option A: dossier Drive déjà décompressé
DRIVE_PROJECT_DIR = DRIVE_ROOT / PROJECT_NAME

# Option B: archive du projet
PROJECT_ZIP = DRIVE_ROOT / f'{PROJECT_NAME}.zip'

# Optionnel: archive séparée contenant data/raw/3DSC_MP.csv et data/raw/cifs/
DATA_ZIP = DRIVE_ROOT / '3DSC_data.zip'

# Réglages rapides pour Colab
EPOCHS = 50
BATCH_SIZE = 32
LEARNING_RATE = 1e-4

print('Drive project dir:', DRIVE_PROJECT_DIR)
print('Project zip:', PROJECT_ZIP)
print('Data zip:', DATA_ZIP)

Drive project dir: /content/drive/MyDrive/Crys_JEPA
Project zip: /content/drive/MyDrive/Crys_JEPA.zip
Data zip: /content/drive/MyDrive/3DSC_data.zip


In [ ]:
# Installer les dépendances minimales pour le MVP DFT/DFT-JEPA
# Colab fournit déjà souvent torch+CUDA; on évite de le réinstaller si possible.
import importlib.util

missing = []
for pkg, import_name in [
    ('pandas', 'pandas'),
    ('numpy', 'numpy'),
    ('pyyaml', 'yaml'),
    ('easydict', 'easydict'),
    ('tqdm', 'tqdm'),
    ('pytest', 'pytest'),
    ('pymatgen', 'pymatgen'),
]:
    if importlib.util.find_spec(import_name) is None:
        missing.append(pkg)

if missing:
    !pip install -q {' '.join(missing)}
else:
    print('Minimal dependencies already installed')

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

Minimal dependencies already installed
torch: 2.11.0+cu128
cuda available: True
gpu: Tesla T4


In [ ]:
# Télécharger le jeu de données 3DSC depuis aimat-lab/3DSC (GitHub)
import urllib.request
import zipfile
from pathlib import Path

# Chemins absolus pour être indépendant du répertoire de travail courant
csv_path = PROJECT_DIR / 'data/raw/3DSC_MP.csv'
cif_dir  = PROJECT_DIR / 'data/raw/cifs'

n_cifs = len(list(cif_dir.glob('*.cif'))) if cif_dir.exists() else 0
if csv_path.exists() and n_cifs > 0:
    print(f'Données déjà présentes — CSV: {csv_path}, CIFs: {n_cifs}')
else:
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    cif_dir.mkdir(parents=True, exist_ok=True)

    # 1. CSV en téléchargement direct
    if not csv_path.exists():
        CSV_URL = (
            'https://raw.githubusercontent.com/aimat-lab/3DSC/main/'
            'superconductors_3D/data/final/MP/3DSC_MP.csv'
        )
        print('Téléchargement du CSV...')
        urllib.request.urlretrieve(CSV_URL, csv_path)
        print(f'  CSV: {csv_path} ({csv_path.stat().st_size // 1024} Ko)')
    else:
        print('CSV déjà présent.')

    # 2. CIFs via l'archive zip du repo (~40 Mo)
    if n_cifs == 0:
        REPO_ZIP_URL = 'https://github.com/aimat-lab/3DSC/archive/refs/heads/main.zip'
        zip_path = PROJECT_DIR / 'data/raw/3DSC_repo.zip'

        print('Téléchargement du repo aimat-lab/3DSC (~40 Mo)...')
        urllib.request.urlretrieve(REPO_ZIP_URL, zip_path)
        print(f'  Archive: {zip_path.stat().st_size // (1024**2)} Mo')

        CIFS_MARKER = 'superconductors_3D/data/final/MP/cifs/'

        print('Extraction des CIFs...')
        n_extracted = 0
        with zipfile.ZipFile(zip_path, 'r') as zf:
            all_entries = zf.namelist()
            print('  Premières entrées zip:', all_entries[:3])
            for entry in all_entries:
                if CIFS_MARKER in entry and entry.endswith('.cif'):
                    target = cif_dir / Path(entry).name
                    with zf.open(entry) as src, open(target, 'wb') as dst:
                        dst.write(src.read())
                    n_extracted += 1

        zip_path.unlink()
        print(f'  {n_extracted} CIFs extraits dans {cif_dir}')
    else:
        print(f'CIFs déjà présents ({n_cifs} fichiers).')

    print()
    print('CSV exists:', csv_path.exists())
    print('CIF count:', len(list(cif_dir.glob('*.cif'))))

CSV déjà présent.
Téléchargement du repo aimat-lab/3DSC (~40 Mo)...
  Archive: 17 Mo
Extraction des CIFs...
  Premières entrées zip: ['3DSC-main/', '3DSC-main/.gitignore', '3DSC-main/DATASET_STATISTICS.md']
  10904 CIFs extraits dans /content/Crys_JEPA/data/raw/cifs

CSV exists: True
CIF count: 10904


In [ ]:
# Décompresser les données si elles ne sont pas déjà dans le projet
from pathlib import Path
import zipfile

csv_path = PROJECT_DIR / 'data/raw/3DSC_MP.csv'
cif_dir  = PROJECT_DIR / 'data/raw/cifs'

if (not csv_path.exists() or not cif_dir.exists() or len(list(cif_dir.glob('*.cif'))) == 0) and DATA_ZIP.exists():
    with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
        zf.extractall(PROJECT_DIR)

print('CSV exists:', csv_path.exists(), csv_path)
print('CIF dir exists:', cif_dir.exists(), cif_dir)
print('Number of CIF files:', len(list(cif_dir.glob('*.cif'))) if cif_dir.exists() else 0)

if not csv_path.exists():
    raise FileNotFoundError('Missing data/raw/3DSC_MP.csv')
if not cif_dir.exists() or len(list(cif_dir.glob('*.cif'))) == 0:
    raise FileNotFoundError('Missing CIF files in data/raw/cifs')

CSV exists: True /content/Crys_JEPA/data/raw/3DSC_MP.csv
CIF dir exists: True /content/Crys_JEPA/data/raw/cifs
Number of CIF files: 10904


In [ ]:
# Fixer le répertoire de travail et ajuster les configs pour Colab
import os
import yaml
from pathlib import Path

os.chdir(PROJECT_DIR)
print('Working directory:', Path.cwd())

CONFIGS = {
    'dft': PROJECT_DIR / 'configs/ablation/dft.yaml',
    'crys_jepa_dft': PROJECT_DIR / 'configs/ablation/crys_jepa_dft.yaml',
}

for name, cfg_path in CONFIGS.items():
    with cfg_path.open('r', encoding='utf-8') as f:
        cfg = yaml.safe_load(f)
    cfg['training']['epochs'] = int(EPOCHS)
    cfg['training']['batch_size'] = int(BATCH_SIZE)
    cfg['training']['learning_rate'] = float(LEARNING_RATE)
    cfg['training']['device'] = 'cuda'
    cfg['checkpoints']['save_dir'] = f'checkpoints/colab_{name}'
    with cfg_path.open('w', encoding='utf-8') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    print(name, cfg_path)
    print('  save_dir:', cfg['checkpoints']['save_dir'])
    print('  input_mode:', cfg['model']['input_mode'])

In [ ]:
# Lancer les tests rapides du MVP
!python -m pytest tests/test_superconductivity_mvp.py -q

In [ ]:
# Entraîner DFT seul
!python scripts/train_mvp.py --config configs/ablation/dft.yaml

In [ ]:
# Entraîner DFT-JEPA
!python scripts/train_mvp.py --config configs/ablation/crys_jepa_dft.yaml

In [ ]:
# Évaluer explicitement et sauvegarder les prédictions
!python scripts/evaluate_mvp.py --config configs/ablation/dft.yaml --checkpoint checkpoints/colab_dft/best.pt --predictions-csv predictions_dft.csv
!python scripts/evaluate_mvp.py --config configs/ablation/crys_jepa_dft.yaml --checkpoint checkpoints/colab_crys_jepa_dft/best.pt --predictions-csv predictions_crys_jepa_dft.csv

In [ ]:
# Construire un tableau comparatif depuis les checkpoints
import torch
import pandas as pd
from pathlib import Path

rows = []
for mode, ckpt_path in {
    'dft': 'checkpoints/colab_dft/best.pt',
    'crys_jepa_dft': 'checkpoints/colab_crys_jepa_dft/best.pt',
}.items():
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    metrics = ckpt.get('val_metrics', {})
    cls = metrics.get('classification', {})
    reg = metrics.get('regression', {})
    rows.append({
        'mode': mode,
        'epoch': ckpt.get('epoch'),
        'val_accuracy': cls.get('accuracy'),
        'val_f1': cls.get('f1'),
        'val_roc_auc': cls.get('roc_auc'),
        'val_mae': reg.get('mae'),
        'val_rmse': reg.get('rmse'),
        'val_mae_superconductors': reg.get('mae_superconductors'),
        'val_mae_high_tc': reg.get('mae_high_tc'),
    })

summary = pd.DataFrame(rows)
summary.to_csv('colab_ablation_validation_summary.csv', index=False)
summary

In [ ]:
# Sauvegarder checkpoints, prédictions et résumé vers Drive
import shutil
from pathlib import Path

OUT_DIR = DRIVE_ROOT / 'Crys_JEPA_colab_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

for rel in [
    'checkpoints/colab_dft',
    'checkpoints/colab_crys_jepa_dft',
    'predictions_dft.csv',
    'predictions_crys_jepa_dft.csv',
    'colab_ablation_validation_summary.csv',
]:
    src = PROJECT_DIR / rel
    dst = OUT_DIR / Path(rel).name
    if src.is_dir():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    elif src.exists():
        shutil.copy2(src, dst)
    print('saved:', dst)

print('Outputs:', OUT_DIR)

## Notes

- Le modèle `crys_jepa_dft` utilise la fusion tardive: embedding structurel + MLP DFT.
- Dans ce MVP, si aucun checkpoint Crys-JEPA réel n'est indiqué, l'encodeur structurel reste le placeholder compatible déjà utilisé dans le projet.
- Pour utiliser un vrai checkpoint Crys-JEPA, renseigne `model.crys_jepa_checkpoint` dans la config avant l'entraînement.